# Customer Data Cleaning and Preprocessing

**Goal:** turn the supplied customer CSV into a consistent dataset without silently discarding otherwise usable records.

## Cleaning steps
- Normalize column text, names, email casing, country names, customer segments and activity flags.
- Parse mixed date formats while documenting the assumption for ambiguous slash-formatted dates.
- Convert spend values to numeric and treat negative sentinel values as missing.
- Remove exact duplicate rows only; retain duplicate customer IDs for review rather than deleting records without a rule.
- Report data-quality checks and export the cleaned data.

**Input:** `dirty-customers.csv`  
**Output:** `cleaned_data.csv`

Run the notebook from this folder so the relative paths resolve.

In [ ]:
import re
import numpy as np
import pandas as pd

INPUT_FILE = "dirty-customers.csv"
OUTPUT_FILE = "cleaned_data.csv"

df = pd.read_csv(INPUT_FILE, dtype={"customer_id": "string"})
print("Original shape:", df.shape)
display(df.head())

## 1. Normalize missing values and remove exact duplicate rows

The earlier version called `dropna()` before `fillna()`, which removed rows before missing values could be handled. This version keeps rows and treats each field according to its meaning.

In [ ]:
cleaned = df.copy()

# Standardize whitespace and common text representations of missing values.
missing_tokens = {"", "null", "none", "nan", "n/a", "na", "-"}
for col in cleaned.select_dtypes(include=["object", "string"]).columns:
    values = cleaned[col].astype("string").str.strip()
    cleaned[col] = values.mask(values.str.lower().isin(missing_tokens))

before_exact_duplicates = len(cleaned)
cleaned = cleaned.drop_duplicates().copy()
exact_duplicates_removed = before_exact_duplicates - len(cleaned)

# Keep IDs as text so any leading zeroes are preserved.
cleaned["customer_id"] = cleaned["customer_id"].astype("string").str.strip()
cleaned["full_name"] = cleaned["full_name"].astype("string").str.strip().str.title()
cleaned["email"] = cleaned["email"].astype("string").str.strip().str.lower()

## 2. Standardize countries, segments and activity status

Unknown or unrecognized values are retained as `Unknown` / missing rather than guessed.

In [ ]:
country_map = {
    "US": "United States", "USA": "United States", "U.S.": "United States",
    "GB": "United Kingdom", "UK": "United Kingdom",
    "DE": "Germany", "FR": "France", "BR": "Brazil"
}
country = cleaned["country"].astype("string").str.strip().str.upper()
cleaned["country"] = country.map(country_map).fillna(country.str.title()).fillna("Unknown")

segment_map = {
    "ENT": "Enterprise", "ENTERPRISE": "Enterprise",
    "SMB": "Small Business", "SMB ": "Small Business",
    "SMALL BUSINESS": "Small Business",
    "MIDMARKET": "Mid-Market", "MID MARKET": "Mid-Market",
    "MID-MARKET": "Mid-Market", "CONSUMER": "Consumer"
}
segment = cleaned["segment"].astype("string").str.strip().str.upper()
cleaned["segment"] = segment.map(segment_map).fillna(segment.str.title()).fillna("Unknown")

active_map = {
    "1": True, "TRUE": True, "YES": True, "Y": True,
    "0": False, "FALSE": False, "NO": False, "N": False
}
active = cleaned["is_active"].astype("string").str.strip().str.upper()
cleaned["is_active"] = active.map(active_map).astype("boolean")

display(cleaned[["country", "segment", "is_active"]].head())

## 3. Parse dates and clean spend

For ambiguous numeric dates such as `04/05/2023`, this notebook assumes **month/day/year**. Dot-separated dates such as `08.01.2023` are treated as **day.month.year**. Missing or unparseable dates remain missing.

Negative spend values are treated as invalid sentinels. Missing spend values are imputed with the dataset median; the median is reported below.

In [ ]:
def parse_signup_date(value):
    if pd.isna(value):
        return pd.NaT
    text = str(value).strip()
    if not text or text.lower() in missing_tokens:
        return pd.NaT

    if re.fullmatch(r"\d{4}-\d{1,2}-\d{1,2}", text):
        return pd.to_datetime(text, format="%Y-%m-%d", errors="coerce")
    if re.fullmatch(r"\d{1,2}\.\d{1,2}\.\d{4}", text):
        return pd.to_datetime(text, format="%d.%m.%Y", errors="coerce")
    if re.fullmatch(r"\d{1,2}/\d{1,2}/\d{4}", text):
        first, second, _ = map(int, text.split("/"))
        fmt = "%d/%m/%Y" if first > 12 else "%m/%d/%Y"
        return pd.to_datetime(text, format=fmt, errors="coerce")
    return pd.to_datetime(text, errors="coerce")

cleaned["signup_date"] = cleaned["signup_date"].apply(parse_signup_date)

spend_text = (
    cleaned["spend"].astype("string")
    .str.replace(r"(?i)USD|\$|,", "", regex=True)
    .str.strip()
)
cleaned["spend"] = pd.to_numeric(spend_text, errors="coerce")
cleaned.loc[cleaned["spend"] < 0, "spend"] = np.nan

spend_median = cleaned["spend"].median()
if pd.notna(spend_median):
    cleaned["spend"] = cleaned["spend"].fillna(spend_median)

print("Spend median used for imputation:", spend_median)
print("Unparsed / missing signup dates:", cleaned["signup_date"].isna().sum())

## 4. Validate and export

Email addresses are not rewritten when malformed; they are flagged for review. Duplicate customer IDs are reported but not removed because the data alone does not establish which record should be retained.

In [ ]:
email_pattern = r"^[A-Za-z0-9.!#$%&'*+/=?^_\x60{|}~-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
cleaned["email_valid"] = cleaned["email"].str.match(email_pattern, na=False)

duplicate_customer_ids = cleaned["customer_id"].duplicated(keep=False).sum()
missing_after = cleaned.isna().sum().sort_values(ascending=False)

print("Rows after cleaning:", len(cleaned))
print("Exact duplicate rows removed:", exact_duplicates_removed)
print("Rows with duplicate customer IDs (retained for review):", duplicate_customer_ids)
print("\nMissing values after cleaning:")
display(missing_after.to_frame("missing_count"))

print("\nInvalid or missing emails:", (~cleaned["email_valid"]).sum())
display(cleaned.head())

cleaned.to_csv(OUTPUT_FILE, index=False)
print(f"Saved cleaned dataset to {OUTPUT_FILE}")